# Assignment 2 — DataQualityAgent

Этот ноутбук показывает полный цикл quality-агента:

- `detect_issues()`
- `choose_strategy()`
- `fix()`
- `compare()`
- `run()` как observe → decide → act → evaluate

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agents").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from agents.data_quality_agent import DataQualityAgent

candidate_paths = [
    ROOT / "data/interim/rewrite.parquet",
    ROOT / "data/raw/merged_raw.csv",
]
for path in candidate_paths:
    if path.exists():
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        break
else:
    raise FileNotFoundError("No input dataset found in data/interim/rewrite.parquet or data/raw/merged_raw.csv")

agent = DataQualityAgent(task_type="text_classification")
report = agent.detect_issues(df)
strategy = agent.choose_strategy(report, df)

df.shape, strategy

In [ ]:
missing_df = pd.DataFrame(
    sorted((report["missing"]["per_column"] or {}).items(), key=lambda x: -x[1]),
    columns=["column", "missing_count"],
)

duplicate_subset = [col for col in agent.duplicate_subset if col in df.columns]
duplicate_mask = df.duplicated(subset=duplicate_subset, keep=False) if duplicate_subset else pd.Series(False, index=df.index)

text_col = agent.text_column
label_col = agent.label_column
text_len_chars = (
    df[text_col].astype("string").fillna("").str.len()
    if text_col in df.columns
    else pd.Series(dtype="int64")
)
text_len_words = (
    df[text_col].astype("string").fillna("").str.split().str.len()
    if text_col in df.columns
    else pd.Series(dtype="int64")
)

quality_overview = pd.DataFrame(
    [
        {"metric": "rows", "value": len(df)},
        {"metric": "duplicate_rows", "value": int(report.get("duplicates") or 0)},
        {"metric": "columns_with_missing", "value": int((missing_df["missing_count"] > 0).sum())},
        {"metric": "outlier_features", "value": len((report.get("outliers") or {}).get("by_feature") or {})},
    ]
)

quality_overview

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

if not missing_df.empty:
    missing_df.plot(kind="bar", x="column", y="missing_count", ax=axes[0], color="goldenrod")
else:
    axes[0].text(0.5, 0.5, "No missing values", ha="center", va="center")
axes[0].set_title("Missing values")
axes[0].set_ylabel("count")

imbalance = report.get("imbalance") or {}
counts = pd.Series(imbalance.get("counts") or {})
if not counts.empty:
    counts.sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="coral")
else:
    axes[1].text(0.5, 0.5, "No label distribution", ha="center", va="center")
axes[1].set_title("Class imbalance")
axes[1].set_ylabel("count")

pd.Series(
    {
        "unique_rows": len(df) - int(report.get("duplicates") or 0),
        "duplicate_rows": int(report.get("duplicates") or 0),
    }
).plot(kind="bar", ax=axes[2], color=["seagreen", "indianred"])
axes[2].set_title("Duplicate rows")
axes[2].set_ylabel("count")

if not text_len_chars.empty:
    axes[3].boxplot(
        [text_len_chars, text_len_words],
        tick_labels=["chars", "words"],
    )
else:
    axes[3].text(0.5, 0.5, "No text column", ha="center", va="center")
axes[3].set_title("Text length outliers")
axes[3].set_ylabel("length")

plt.tight_layout()
plt.show()

In [ ]:
outlier_detail = (report.get("outliers") or {}).get("by_feature") or {}
outlier_df = pd.DataFrame(
    [
        {
            "feature": feature,
            "n_iqr": detail.get("n_iqr", 0),
            "iqr_low": (detail.get("iqr_bounds") or [None, None])[0],
            "iqr_high": (detail.get("iqr_bounds") or [None, None])[1],
        }
        for feature, detail in outlier_detail.items()
    ]
)

preview_cols = [col for col in [text_col, label_col] if col in df.columns]
duplicate_examples = df.loc[duplicate_mask, preview_cols].head(10) if duplicate_subset else pd.DataFrame()

outlier_mask = pd.Series(False, index=df.index)
char_bounds = outlier_detail.get("text_len_chars", {}).get("iqr_bounds")
word_bounds = outlier_detail.get("text_len_words", {}).get("iqr_bounds")
if char_bounds:
    outlier_mask |= (text_len_chars < char_bounds[0]) | (text_len_chars > char_bounds[1])
if word_bounds:
    outlier_mask |= (text_len_words < word_bounds[0]) | (text_len_words > word_bounds[1])
outlier_examples = df.loc[outlier_mask, preview_cols].head(10)

print("Dataset overview")
display(quality_overview)
print("Outlier summary by feature")
display(outlier_df)
print("Duplicate text examples")
display(duplicate_examples)
print("Outlier text examples")
display(outlier_examples)

In [ ]:
strategy_drop = {
    "missing": "fill",
    "duplicates": "drop",
    "outliers": "drop_iqr",
}
strategy_clip = {
    "missing": "fill",
    "duplicates": "drop",
    "outliers": "clip_iqr",
}

df_drop = agent.fix(df, strategy_drop)
df_clip = agent.fix(df, strategy_clip)
comparison_drop = agent.compare(df, df_drop)
comparison_clip = agent.compare(df, df_clip)

strategy_summary = pd.DataFrame(
    [
        {
            "strategy": "drop_iqr",
            "rows_after": len(df_drop),
            "duplicates_after": int(df_drop.duplicated(subset=duplicate_subset).sum()) if duplicate_subset else 0,
            "missing_text_after": int(df_drop[agent.text_column].isna().sum()) if agent.text_column in df_drop.columns else 0,
        },
        {
            "strategy": "clip_iqr",
            "rows_after": len(df_clip),
            "duplicates_after": int(df_clip.duplicated(subset=duplicate_subset).sum()) if duplicate_subset else 0,
            "missing_text_after": int(df_clip[agent.text_column].isna().sum()) if agent.text_column in df_clip.columns else 0,
        },
    ]
)

print("Agent reasoning:")
for line in strategy["reasoning"]:
    print("-", line)

strategy_summary

In [ ]:
result = agent.run(df)
comparison_agent = result["comparison"].assign(strategy="agent_default")
comparison_drop = comparison_drop.assign(strategy="drop_iqr")
comparison_clip = comparison_clip.assign(strategy="clip_iqr")

comparison_all = pd.concat([comparison_drop, comparison_clip, comparison_agent], ignore_index=True)
comparison_all.pivot(index="metric", columns="strategy", values="after")

## Обоснование лучшей стратегии

Для задачи классификации коротких и средних пользовательских запросов критично удалить строки без `text` и без `label`, потому что такие наблюдения не несут полезного сигнала для обучения и искажают оценку качества. Дубликаты по `text` тоже нежелательны: они завышают частоту отдельных формулировок и могут искусственно улучшать метрики модели, если почти одинаковые строки попадают и в train, и в test.

Для текстовой задачи выбросы разумно искать прежде всего по длине текста. Очень длинные строки в таком benchmark чаще оказываются шумом, donor-only материалом, HTML-фрагментами или нетипичными для prompt classification примерами. Поэтому из двух стратегий более надёжной выглядит `drop_iqr`: она удаляет экстремальные по длине наблюдения целиком, не искажая текст. В отличие от неё, `clip_iqr` укорачивает текст и может повредить смысл, что для NLP-задачи опаснее, чем для числовых признаков. Поэтому в качестве лучшего подхода для этого benchmark логично выбрать стратегию с `duplicates="drop"` и `outliers="drop_iqr"`. 